# 10.4 Writing data to relational tables

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To use an external relational `table` for writing only, you should employ a `table` declaration
that specifies its read/write status to be OUT. The general form of such a declaration
is

```ampl
table table-name OUT string-list opt :
    key-spec, data-spec, data-spec, ... ;
```

where the optional string-list is specific to the database type and access method being
used. (Again, most subsequent examples do not include a string-list.) AMPL expression
values are subsequently written to the `table` by the command

```ampl
write table table-name ;
```

which uses the `table` declaration that defined `table`-name to determine the information
to be written.

A `table` declaration for writing specifies an external file and possibly a relational
`table` within that file, either explicitly in the string-list or implicitly by default rules. Normally
the named external file or `table` is created if it does not exist, or is overwritten otherwise.
To specify that instead certain columns are to be replaced or are to be added to a
`table`, the `table` declaration must incorporate one or more `data`-specs that have
read/write status IN or INOUT, as discussed in Section 10.5. A specific `table` handler
may also have its own more detailed rules for determining when files and tables are modified
or overwritten, as explained in its documentation.

The key-specs and `data`-specs of `table` declarations for writing external tables superficially
resemble those for reading. The range of AMPL expressions allowed when writing
is much broader, however, including essentially all set-valued and numeric-valued
expressions. Moreover, whereas the `table` rows to be read are those of some existing
`table`, the rows to be written must be determined from AMPL expressions in some part of
a `table` declaration. Specifically, rows to be written can be inferred either from the
`data`-specs, using the same conventions as in `display` commands, or from the key-spec.
Each of these alternatives employs a characteristic `table` syntax as described below.

Writing rows inferred from the `data` specifications
If the key-spec is simply a bracketed list of the names of key columns,

```ampl
[key-col-name, key-col-name, ...]
```

the `table` declaration works much like the `display` command. It determines the
external `table` rows to be written by taking the union of the indexing sets stated or implied
in the `data`-specs. The format of the `data`-spec list is the same as in `display`, except
that all of the items listed must have the same dimension.

In the simplest case, the `data`-specs are the names of `model` components indexed over
the same set:

```ampl
table Foods OUT: [FoodName], f_min, Buy, f_max;
```

When `write table` Foods is executed, it creates a key column FoodName and `data`
columns f_min, Buy, and f_max. Since the AMPL components corresponding to the
`data` columns are all indexed over the AMPL set FOOD, one row is created for each member
of FOOD. In a representative row, a member of FOOD is written to the key column
FoodName, and the values of f_min, Buy, and f_max subscripted by that member are
written to the like-named `data` columns. For the `data` used in the diet example, the resulting
relational `table` would be:

```ampl
FoodName  f_min    Buy      f_max
BEEF       2      5.36061    10
CHK        2      2          10
FISH       2      2          10
HAM        2     10          10
MCH        2     10          10
MTL        2     10          10
SPG        2      9.30605    10
TUR        2      2          10
```

Tables corresponding to higher-dimensional sets are handled analogously, with the number
of bracketed key-column names listed in the key-spec being equal to the dimension of
the items in the `data`-spec. Thus a `table` containing the results from steelT.mod could
be defined as

```ampl
table SteelProd OUT: [PROD, TIME], Make, Sell, Inv;
```

Because Make and Sell are indexed over {PROD,1..T}, while Inv is indexed over
{PROD,0..T}, a subsequent `write table` SteelProd command would produce a
`table` with one row for each member of the union of these sets:

```ampl
PROD   TIME   Make   Sell   Inv
bands    0      .      .     10
bands    1    5990   6000     0
bands    2    6000   6000     0
bands    3    1400   1400     0
bands    4    2000   2000     0
coils    0      .      .      0
coils    1    1407    307  1100
coils    2    1400   2500     0
coils    3    3500   3500     0
coils    4    4200   4200     0
```

Two rows are empty in the columns for Make and Sell, because ("bands",0) and
("coils",0) are not members of the index sets of Make and Sell. We use a "."
here to indicate the empty `table` entries, but the actual appearance and handling of empty
entries will vary depending on the database software being used.

If this form is applied to writing suffixed variable or constraint names, such as the
dual and slack values related to the constraint Diet:

```ampl
table Nutrs OUT: [Nutrient],
   Diet.lslack, Diet.ldual, Diet.uslack, Diet.udual; # ERROR
```

a subsequent `write table` Nutrs command is likely to be rejected, because names
with a "dot" in the middle are not allowed as column names by most database software:

```ampl
ampl: write table Nutrs;
Error executing "write table" command:
   Error writing table Nutrs with table handler ampl.odbc:
   Column 2's name "Diet.lslack" contains non-alphanumeric
      character '.'.
```

This situation requires that each AMPL expression be followed by the operator ˜ and a
corresponding valid column name for use in the relational `table`:

```ampl
table Nutrs OUT: [Nutrient],
   Diet.lslack ˜ lb_slack, Diet.ldual ˜ lb_dual,
   Diet.uslack ˜ ub_slack, Diet.udual ˜ ub_dual;
```

This says that the values represented by Diet.lslack should be placed in a column
named lb_slack, the values represented by Diet.ldual should be placed in a column
named lb_dual, and so forth. With the `table` defined in this way, a write
`table` Nutrs command produces the intended relational `table`:

```ampl
Nutrient   lb_slack     lb_dual     ub_slack     ub_dual
   A        1256.29     0           18043.7       0
   B1        336.257    0           18963.7       0
   B2          0        0.404585    19300         0
   C         982.515    0           18317.5       0
   CAL      3794.62     0            4205.38      0
   NA      50000        0               0        -0.00306905
```

The ˜ can also be used with unsuffixed names, if it is desired to assign the dabatase column
a name different from the corresponding AMPL entity.

More general expressions for the values in `data` columns require the use of dummy
indices, in the same way that they are used in the `data`-list of a `display` command.
Since indexed AMPL expressions are rarely valid column names for a database, they
should generally be followed by ˜ `data`-col-name to provide a valid name for the corresponding
relational `table` column that is to be written. To write a column servings
containing the number of servings of each food to be bought and a column percent
giving the amount bought as a percentage of the maximum allowed, for example, the
`table` declaration could be given as either

```ampl
table Purchases OUT: [FoodName],
   Buy ˜ servings, {j in FOOD} 100*Buy[j]/f_max[j] ˜ percent;
```

or

```ampl
table Purchases OUT: [FoodName],
   {j in FOOD} (Buy[j] ˜ servings,
                100*Buy[j]/f_max[j] ˜ percent);
```

Either way, since both `data`-specs give expressions indexed over the AMPL set FOOD, the
resulting `table` has one row for each member of that set:

```ampl
FoodName  servings    percent
BEEF       5.36061     53.6061
CHK        2           20
FISH       2           20
HAM       10          100
MCH       10          100
MTL       10          100
SPG        9.30605     93.0605
TUR        2           20
```

The expression in a `data`-spec may also use operators like sum that define their own
dummy indices. Thus a `table` of total production and sales by period for steelT.mod
could be specified by

```ampl
table SteelTotal OUT: [TIME],
   {t in 1..T} (sum {p in PROD} Make[p,t] ˜ Made,
                sum {p in PROD} Sell[p,t] ˜ Sold);
```

As a two-dimensional example, a `table` of the amounts sold and the fractions of demand
met could be specified by

```ampl
table SteelSales OUT: [PROD, TIME], Sell,
   {p in PROD, t in 1..T} Sell[p,t]/market[p,t] ˜ FracDemand;
```

The resulting external `table` would have key columns PROD and TIME, and `data` columns
Sell and FracDemand.

Writing rows inferred from a key specification
An alternative form of `table` declaration specifies that one `table` row is to be written
for each member of an explicitly specified AMPL set. For the declaration to work in this
way, the key-spec must be written as

```ampl
set-spec -> [key-col-spec, key-col-spec, ...]
```

In contrast to the arrow <- that points from a key-column list to an AMPL set, indicating
values to be read into the set, this form uses an arrow -> that points from an AMPL set to
a key column list, indicating information to be written from the set into the key columns.

An explicit expression for the row index set is given by the set-spec, which can be the
name of an AMPL set, or any AMPL set-expression enclosed in braces { }. The key-colspecs
give the names of the corresponding key columns in the database. Dummy indices,
if needed, can appear either with the set-spec or the key-col-specs, as we will show.

The simplest case of this form involves writing database columns for `model` components
indexed over the same one-dimensional set, as in this example for diet.mod:

```ampl
table Foods OUT: FOOD -> [FoodName], f_min, Buy, f_max;
```

When `write table` Foods is executed, a `table` row is created for each member of the
AMPL set FOOD. In that row, the set member is written to the key column FoodName,
and the values of f_min, Buy, and f_max subscripted by the set member are written to
the like-named `data` columns. (For the `data` used in our diet example, the resulting `table`
would be the same as for the FoodName `table` given previously in this section.) If the
key column has the same name, FOOD, as the AMPL set, the appropriate `table` declaration
becomes

```ampl
table Foods OUT: FOOD -> [FOOD], f_min, Buy, f_max;
```

In this special case only, the key-spec can also be written in the abbreviated form
[FOOD] OUT.

The use of ˜ with AMPL names and suffixed names is governed by the considerations
previously described, so that the example of diet slack and dual values would be written

```ampl
table Nutrs OUT: NUTR -> [Nutrient],
   Diet.lslack ˜ lb_slack, Diet.ldual ˜ lb_dual,
   Diet.uslack ˜ ub_slack, Diet.udual ˜ ub_dual;
```

and `write table` Nutrs would give the same `table` as previously shown.

More general expressions for the values in `data` columns require the use of dummy
indices. Since the rows to be written are determined from the key-spec, however, the
dummies are also defined there (rather than in the `data`-specs as in the alternative form
above). To specify a column containing the amount of a food bought as a percentage of
the maximum allowed, for example, it is necessary to write 100*Buy[j]/f_max[j],
which in turn requires that dummy index `j` be defined. The definition may appear either
in a set-spec of the form { index-list in set-expr }:

```ampl
table Purchases OUT: {j in FOOD} -> [FoodName],
   Buy[j] ˜ servings, 100*Buy[j]/f_max[j] ˜ percent;
```

or in a key-col-spec of the form index ˜ key-col-name:

```ampl
table Purchases OUT: FOOD -> [j ˜ FoodName],
   Buy[j] ˜ servings, 100*Buy[j]/f_max[j] ˜ percent;
```

These two forms are equivalent. Either way, as each row is written, the index `j` takes the
key column value, which is used in interpreting the expressions that give the values for
the `data` columns. For our example, the resulting `table`, having key column FoodName
and `data` columns servings and percent, is the same as previously shown. Similarly,
the previous example of the `table` SteelTotal could be written as either

```ampl
table SteelTotal OUT: {t in 1..T} -> [TIME],
   sum {p in PROD} Make[p,t] ˜ Made,
   sum {p in PROD} Sell[p,t] ˜ Sold;
```

or

```ampl
table SteelTotal OUT: {1..T} -> [t ˜ TIME],
   sum {p in PROD} Make[p,t] ˜ Made,
   sum {p in PROD} Sell[p,t] ˜ Sold;
```

The result will have a key column TIME containing the integers 1 through T, and `data`
columns Made and Sold containing the values of the two summations. (Notice that
since 1..T is a set-expression, rather than the name of a set, it must be included in
braces to be used as a set-spec.)

Tables corresponding to higher-dimensional sets are handled analogously, with the
number of key-col-specs listed in brackets being equal to the dimension of the set-spec.
Thus a `table` containing the results from steelT.mod could be defined as

```ampl
table SteelProd OUT:
   {PROD, 1..T} -> [PROD, TIME], Make, Sell, Inv;
```

and a subsequent `write table` SteelProd would produce a `table` of the form

```ampl
PROD     TIME      Make     Sell     Inv
bands      1       5990     6000       0
bands      2       6000     6000       0
bands      3       1400     1400       0
bands      4       2000     2000       0
coils      1       1407      307    1100
coils      2       1400     2500       0
coils      3       3500     3500       0
       coils      4       4200     4200       0
```

This result is not quite the same as the `table` produced by the previous SteelProd
example, because the rows to be written here correspond explicitly to the members of the
set {PROD, 1..T}, rather than being inferred from the indexing sets of Make, Sell,
and Inv. In particular, the values of Inv["bands",0] and Inv["coils",0] do
not appear in this `table`.

The options for dummy indices in higher dimensions are the same as in one dimension.
Thus our example SteelSales could be written either using dummy indices
defined in the set-spec:

```ampl
table SteelSales OUT:
   {p in PROD, t in 1..T} -> [PROD, TIME],
   Sell[p,t] ˜ sold, Sell[p,t]/market[p,t] ˜ met;
```

or with dummy indices added to the key-col-specs:

```ampl
table SteelSales OUT:
   {PROD,1..T} -> [p ˜ PROD, t ˜ TIME],
   Sell[p,t] ˜ sold, Sell[p,t]/market[p,t] ˜ met;
```

If dummy indices happen to appear in both the set-spec and the key-col-specs, ones in the
key-col-specs take precedence.